In [0]:
# %pip install folium

In [0]:
EH_NAMESPACE = '<namespace>.servicebus.windows.net'
EH_NAME = '<event-hub-name>'
SERVICE_CREDENTIAL = '<service-credential>'

CATALOG = spark.sql('SELECT current_catalog()').first()[0]
SCHEMA = 'iss_tracker'

BRONZE = f'{SCHEMA}.bronze_events'
SILVER = f'{SCHEMA}.silver_positions'
GOLD = f'{SCHEMA}.gold_positions'

CKPT_ROOT = f'/Volumes/{CATALOG}/{SCHEMA}/pipeline_state/checkpoints'
CKPT_BRONZE = f'{CKPT_ROOT}/bronze_v1'
CKPT_SILVER = f'{CKPT_ROOT}/silver_v2'

KAFKA_OPTIONS = {
    'kafka.bootstrap.servers': f'{EH_NAMESPACE}:9093',
    'subscribe': EH_NAME,
    'databricks.serviceCredential': SERVICE_CREDENTIAL
}

print('CATALOG    :', CATALOG)
print('Hub        :', EH_NAMESPACE, '/', EH_NAME)
print('Checkpoints:', CKPT_ROOT)

In [0]:
from pyspark.sql.functions import col, lit, when, current_timestamp, min as fmin
from pyspark.sql.window import Window

PERIOD = 5574

silver = spark.table(SILVER).dropDuplicates(['event_ts'])
w = Window.partitionBy(lit(1))

gold = (silver
        .withColumn('orbit_number',
                    ((col('event_ts') - fmin('event_ts').over(w))/PERIOD).cast('int') + 1)
        .withColumn('hemisphere', when(col('lat') >=0, 'Northern').otherwise('Southern'))
        .withColumn('altitude_km', lit(408.0))
        .withColumn('speed_kmh', lit(27600.0))
        .withColumn('processed_at', current_timestamp()))

(gold.write.format('delta').mode('overwrite')
 .option('overwriteSchema', 'true')
 .saveAsTable(GOLD))

gold = spark.table(GOLD)
print('Gold rows: ', gold.count())
gold.orderBy('event_ts').show(5)

In [0]:
from pyspark.sql.functions import count, min, max

gold = spark.table(GOLD)
if gold.count() == 0:
    raise Exception('Gold is empty. Run Bronze and Silver.')

gold.select(count('*').alias('rows'),
            min('event_dt').alias('first'),
            max('event_dt').alias('last')).show(truncate=False)
display(gold.select('lat', 'lon', 'event_dt', 'orbit_number').orderBy('event_ts'))

In [0]:
display(gold.select(col('event_dt').alias('Time'),
                    col('lat').alias('Latitude'))
        .orderBy('event_dt'))

In [0]:
spark.sql(f'''
          SELECT orbit_number,
            COUNT(*) AS readings,
            ROUND(COUNT(*)*10*100.0/{PERIOD}, 1) AS pct_captured,
            MIN(event_dt) AS orbit_start,
            MAX(event_dt) AS orbit_end
          FROM {GOLD}
          GROUP BY orbit_number
          ORDER BY orbit_number
          ''').show(truncate=False)

In [0]:
import folium

positions = (spark.table(GOLD)
             .select('lat', 'lon', 'event_ts', 'orbit_number')
             .orderBy('event_ts')
             .collect())
latest = positions[-1]

m = folium.Map(location=[latest.lat, latest.lon], zoom_start=2, tiles='OpenStreetMap')
# folium.TileLayer(
#     tiles='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
#     attr='&copy: OpenStreetMap contributors &copy; CARTO',
#     name='Dark',
#     subdomains='abcd',
#     max_zoom=20
# ).add_to(m)
colors = ['#00E5FF', '#FF4081', '#69FF47', '#FFEA00', '#FF6D00']

# Split path to prevent lines from crossing at antimeridian, collection gaps and orbit boundaries
segments, seg, prev = [], [], None
for p in positions:
    if prev is not None:
        jump = abs(p.lon - prev.lon) > 180 or p.event_ts - prev.event_ts > 60
        if jump or p.orbit_number != prev.orbit_number:
            segments.append(seg)
            seg = [] if jump else [prev]
    seg.append(p)
    prev = p
segments.append(seg)

for seg in segments:
    if len(seg) < 2:
        continue
    orbit = seg[-1].orbit_number
    folium.PolyLine([[r.lat, r.lon] for r in seg],
                    color=colors[(orbit-1)%len(colors)],
                    weight=2.5,
                    opacity=0.85,
                    tooltip=f'Orbit {orbit}').add_to(m)
folium.CircleMarker([positions[0].lat, positions[0].lon],
                    radius=8,
                    color='#69FF47',
                    fill=True,
                    fill_color='#69FF47',
                    tooltip='Start').add_to(m)
folium.CircleMarker([latest.lat, latest.lon],
                    radius=10,
                    color='#FF4081',
                    fill=True,
                    fill_color='#FF4081',
                    tooltip=f'Latest: {latest.lat:.3f}, {latest.lon:.3f}').add_to(m)
displayHTML(m._repr_html_())

n_orbits = len({p.orbit_number for p in positions})
print(f'{len(positions)} positions across {n_orbits} orbit(s)')

In [0]:
import base64

pdf = spark.table(GOLD).orderBy('event_ts').toPandas()
b64 = base64.b64encode(pdf.to_csv(index=False).encode()).decode()

displayHTML(f'''
            <a href="data:text/csv;base64,{b64}" download="iss_positions.csv" style="display:inline-block;padding:12px 24px;background:#1565C0;color:white;text-decoration:none;border-radius:6px;">Download iss_positions.csv ({len(pdf)} records)</a>
            ''')